# Extract a dataset of activations

`tl.extract_dataset` runs a set of inputs through a model in batches and stacks the activations of the layers you name, in memory by default or on disk with `path=`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from torch import nn

import torchlens as tl


def tiny_model(seed: int = 0) -> nn.Module:
    """A deterministic three-layer MLP."""

    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 2),
    )


torch.manual_seed(0)
model = tiny_model(seed=7).eval()
stimuli = torch.randn(10, 4)

In-memory mode returns a dict of stacked activations, one tensor per requested layer, ordered like the stimuli.

In [ ]:
features = tl.extract_dataset(model, stimuli, layers=["relu_1_2"], batch_size=4, progress=False)
print({name: tuple(tensor.shape) for name, tensor in features.items()})

With `path=`, each batch is written as a shard beside a `manifest.json`, and the call returns the shard paths. `tl.load` reads the folder back; `.activations` holds the stacked tensors.

In [ ]:
with TemporaryDirectory() as tmpdir:
    out_dir = Path(tmpdir) / "tiny_extraction"
    shard_paths = tl.extract_dataset(
        model,
        stimuli,
        layers=["relu_1_2"],
        batch_size=4,
        path=out_dir,
        progress=False,
    )
    print("shards written:", len(shard_paths))
    loaded = tl.load(out_dir)
    activations = loaded.activations["relu_1_2"]
    print("loaded:", tuple(activations.shape))
    assert torch.equal(activations, features["relu_1_2"])

Disk runs can resume with `resume=True` (experimental); mismatched inputs or settings are rejected.